In [70]:
experiment = "862_image_7_esp_3_k_3_seg"

In [71]:
# To sort the results.csv by instance_id
CSV_PATH = f"../../results/{experiment}/results.csv"

import pandas as pd
df = pd.read_csv(CSV_PATH)
df["instance_id"] = pd.to_numeric(df["instance_id"], errors="coerce")
df_sorted = df.sort_values(by="instance_id", ascending=True)
df_sorted.to_csv(CSV_PATH, index=False)

In [ ]:
# To Merge the results.csv and instance(vnnlib property) info
import pandas as pd
import re
import numpy as np
import matplotlib.pyplot as plt

results_path = f"../../results/{experiment}/results.csv"
stats_path = f"../../results/{experiment}/input_change_stats.csv"
out_path = f"../../results/{experiment}/combined_results.csv"

df_results = pd.read_csv(results_path)
df_stats   = pd.read_csv(stats_path)

# ============================================================
# 1) Parse vnnlib -> image, segment_index, pattern, k, eps
#    Examples:
#      vnnlib/n01440764_tench_global_k10_eps_0.0001.vnnlib
#      vnnlib/n01440764_tench_seg0_fixmask_k50176_eps_0.0003.vnnlib
#      vnnlib/n01440764_tench_seg0_fixnonmask_k10_eps_0.0001.vnnlib
# ============================================================
VN_RE = re.compile(
    r"(?P<image>n\d+_[^/_]+)_"                         # n01440764_tench
    r"(?:(?P<global>global)|seg(?P<seg>\d+)_(?P<fix>fixmask|fixnonmask))_"  # global OR segX_fix*
    r"k(?P<k>\d+)_"                                    # k10
    r"eps_(?P<eps>[0-9.]+)"                             # eps_0.0001
)

def decode_vnnlib(v: str):
    v = str(v)
    base = v.split("/")[-1]
    base = base.replace(".vnnlib", "")
    m = VN_RE.search(base)
    if not m:
        return pd.Series({"image": np.nan, "segment_index": np.nan, "pattern": np.nan, "k": np.nan, "eps": np.nan})

    image = m.group("image")
    k = int(m.group("k"))
    eps = float(m.group("eps"))

    if m.group("global") == "global":
        segment_index = -1
        pattern = "global"
    else:
        segment_index = int(m.group("seg"))
        fix = m.group("fix")  # fixmask or fixnonmask
        # match your stats naming: fix_mask / fix_nonmask
        pattern = {"fixmask": "fix_mask", "fixnonmask": "fix_nonmask"}[fix]

    return pd.Series({"image": image, "segment_index": segment_index, "pattern": pattern, "k": k, "eps": eps})

decoded = df_results["vnnlib"].apply(decode_vnnlib)
df_results_decoded = pd.concat([df_results.copy(), decoded], axis=1)

# ============================================================
# 2) Clean types + make merge-safe eps key (avoid float glitches)
# ============================================================
for d in (df_results_decoded, df_stats):
    d["k"] = pd.to_numeric(d["k"], errors="coerce").astype("Int64")
    d["segment_index"] = pd.to_numeric(d["segment_index"], errors="coerce").astype("Int64")
    d["eps"] = pd.to_numeric(d["eps"], errors="coerce")

# robust merge key for eps (string with fixed precision)
EPS_DECIMALS = 10
df_results_decoded["eps_key"] = df_results_decoded["eps"].round(EPS_DECIMALS)
df_stats["eps_key"] = df_stats["eps"].round(EPS_DECIMALS)

# ============================================================
# 3) Merge with input_change_stats on decoded keys
# ============================================================
merge_keys = ["image", "segment_index", "pattern", "k", "eps_key"]

df_merged = df_stats.merge(
    df_results_decoded.drop(columns=["eps"]),  # keep eps from stats (or swap if you prefer)
    on=merge_keys,
    how="left",
    suffixes=("", "_res")
)

# (optional) keep a single eps column
df_merged = df_merged.drop(columns=["eps_key"])

# Save
df_merged.to_csv(out_path, index=False)
print("Saved:", out_path)
print("Merged shape:", df_merged.shape)

# Quick sanity check: rows that didn't find a matching results row
missing = df_merged["result"].isna().sum() if "result" in df_merged.columns else None
print("Unmatched rows (result is NaN):", missing)

In [ ]:
# To Combining the results of two csv files
import pandas as pd

df1 = pd.read_csv("../results/vggnet16_benchmark2022_segmented_all/results.csv")
df2 = pd.read_csv("../results/vggnet16_benchmark2022_segmented_all/results_part1_until_5700.csv")

combined = pd.concat([df1, df2], ignore_index=True)

combined.to_csv("../results/vggnet16_benchmark2022_segmented_all/results_part1_until_5700.csv", index=False)

In [ ]:
# To find the index of an image in a folder
import os

def find_image_index(folder, image_name):
    """
    folder: path to directory containing images
    image_name: base name WITHOUT extension (e.g. 'n02113186_Cardigan')
    """

    # list only files (ignore subdirs)
    files = [
        f for f in os.listdir(folder)
        if os.path.isfile(os.path.join(folder, f))
    ]

    # sort for stable ordering
    files = sorted(files)

    # strip extensions
    base_names = [os.path.splitext(f)[0] for f in files]

    if image_name not in base_names:
        raise ValueError(
            f"'{image_name}' not found in {folder}\n"
            f"Example names: {base_names[:10]}"
        )

    idx = base_names.index(image_name)

    return idx


# ------------------------
# EXAMPLE USAGE
# ------------------------

if __name__ == "__main__":

    FOLDER = "../../benchmarks/vggnet16_benchmark2022/imagenet-sample"
    IMAGE  = "n02113186_Cardigan"

    idx = find_image_index(FOLDER, IMAGE)

    print(f"Image '{IMAGE}' index in folder: {idx-1}")

In [ ]:
import pandas as pd
import re
from IPython.display import display, Markdown
import numpy as np

CSV_PATH = f"../../results/{experiment}/results.csv"
df = pd.read_csv(CSV_PATH)

# make sure numeric
df["k"] = pd.to_numeric(df["k"], errors="coerce")
df["eps"] = pd.to_numeric(df["eps"], errors="coerce")

dfs_by_keps = {
    (k, eps): subdf.copy()
    for (k, eps), subdf in df.groupby(["k", "eps"])
}

# Example access:
df_all_0001 = dfs_by_keps[(50176, 0.0001)]
df_all_0003 = dfs_by_keps[(50176, 0.0003)]
df_k10_0001 = dfs_by_keps[(10, 0.0001)]
df_k10_0003 = dfs_by_keps[(10, 0.0003)]

def complete_triplets(df, image_col="image"):
    df = df.copy()
    print("Original shape:", df.shape)

    # --------------------------
    # keep only images with exactly 3 rows
    # --------------------------
    counts = df[image_col].value_counts()
    good_images = counts[counts == 3].index
    df = df[df[image_col].isin(good_images)].copy()

    print("Number of images:", len(good_images))

    return df


df_all_0001 = complete_triplets(df_all_0001)
df_all_0003 = complete_triplets(df_all_0003)
df_k10_0001 = complete_triplets(df_k10_0001)
df_k10_0003 = complete_triplets(df_k10_0003)